# 1. Microstructure Research

流程: Orderbook 散点图 -> Wall Mid -> Normalized 图 -> Return ACF -> 交易分类 -> Adverse Selection

In [61]:
import sys
sys.path.insert(0, '.')

from utils.dataio import load_prices, load_prices_wide, load_trades
from utils.orderbook import compute_wall_mid, compute_spread, return_autocorrelation, adverse_selection, order_interval_distribution, trade_interval_distribution, normalized_level_trade_stats
from utils.viz import plot_orderbook_scatter, plot_autocorrelation, plot_trade_profile, plot_normalized_orderbook, plot_interval_distribution
import polars as pl
import numpy as np

In [62]:
# === 配置 ===
ROUND = 1
DAY = -1
PRODUCT = 'INTARIAN_PEPPER_ROOT'  # 改成你要分析的产品

# 逆向工程过滤开关（按量过滤）
FILTER_BY_VOLUME = False
MIN_ORDER_VOLUME = 10
MIN_TRADE_QUANTITY = 3

## Step 1: 加载数据

In [63]:
prices_long = load_prices(ROUND, DAY)
prices_wide = load_prices_wide(ROUND, DAY)
trades = load_trades(ROUND, DAY)

print('Products:', prices_long['product'].unique().to_list())
print('Prices shape:', prices_long.shape)
print('Trades shape:', trades.shape)
trades.head()

Products: ['INTARIAN_PEPPER_ROOT', 'ASH_COATED_OSMIUM']
Prices shape: (65263, 8)
Trades shape: (760, 7)


timestamp,buyer,seller,product,currency,price,quantity
i64,str,str,str,str,f64,i64
2800,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",9985.0,10
3000,null,null,"""INTARIAN_PEPPER_ROOT""","""XIRECS""",10999.0,8
4200,null,null,"""INTARIAN_PEPPER_ROOT""","""XIRECS""",10995.0,4
5200,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",10004.0,6
5200,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",9986.0,9


## Step 2: Orderbook 散点图（核心可视化）

In [64]:
# 计算 wall mid
wall_mid_df = compute_wall_mid(prices_wide.filter(pl.col('product') == PRODUCT))
wall_mid_df.head()

day,timestamp,product,bid_wall_price,bid_wall_volume,ask_wall_price,ask_wall_volume,wall_mid,raw_mid
i64,i64,str,i64,i64,i64,i64,f64,f64
-1,0,"""INTARIAN_PEPPER_ROOT""",10991,15,11009,15,11000.0,10998.5
-1,100,"""INTARIAN_PEPPER_ROOT""",10991,21,11009,21,11000.0,11000.0
-1,200,"""INTARIAN_PEPPER_ROOT""",10994,10,11009,20,11001.5,11001.5
-1,300,"""INTARIAN_PEPPER_ROOT""",null,0,11009,21,null,11006.0
-1,400,"""INTARIAN_PEPPER_ROOT""",10992,20,11009,20,11000.5,10999.0


In [65]:
# Orderbook 散点图 + wall mid + trades
fig = plot_orderbook_scatter(
    prices_long, trades, product=PRODUCT, day=DAY, wall_mid_df=wall_mid_df,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
)
fig.show()

## Step 3: Normalized Orderbook（去趋势）

In [66]:
fig = plot_normalized_orderbook(
    prices_long, wall_mid_df, product=PRODUCT, day=DAY, trades=trades,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    overlay_wall_mid_trend=True,
)
fig.show()

## Step 4: Spread 统计

In [67]:
spread_df = compute_spread(prices_wide.filter(pl.col('product') == PRODUCT))
print('Wall spread stats:')
print(spread_df['wall_spread'].describe())
print(f"\nWall mid vs Raw mid diff: {(spread_df['wall_mid'] - spread_df['raw_mid']).mean():.4f}")

Wall spread stats:
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 9219.0    │
│ null_count ┆ 781.0     │
│ mean       ┆ 17.263586 │
│ std        ┆ 1.720472  │
│ min        ┆ 2.0       │
│ 25%        ┆ 16.0      │
│ 50%        ┆ 18.0      │
│ 75%        ┆ 18.0      │
│ max        ┆ 19.0      │
└────────────┴───────────┘

Wall mid vs Raw mid diff: 0.0138


## Step 5: Return Autocorrelation

In [68]:
wm = wall_mid_df.sort('timestamp')['wall_mid'].to_numpy()
acf, ci = return_autocorrelation(wm, max_lag=20)

fig = plot_autocorrelation(acf, ci, title=f'{PRODUCT} Return ACF')
fig.show()

# 解读
print(f'ACF(1) = {acf[0]:.4f}, 95% CI = +/-{ci:.4f}')
if abs(acf[0]) < ci:
    print('-> 不可预测，专注做市')
elif acf[0] < -ci:
    print('-> 短期均值回复，可以逆向交易')
else:
    print('-> 短期动量，taker 有 informed signal')

ACF(1) = -0.4634, 95% CI = +/-0.0213
-> 短期均值回复，可以逆向交易


## Step 6: 交易者分类

In [69]:
product_trades = trades.filter(pl.col('product') == PRODUCT)

if 'buyer' in trades.columns:
    fig = plot_trade_profile(trades, product=PRODUCT)
    fig.show()
    
    # 每个 trader 的行为模式
    for trader in product_trades['buyer'].unique().to_list():
        if not trader:
            continue
        tt = product_trades.filter(pl.col('buyer') == trader)
        print(f"\n{trader}: {len(tt)} buys, qty mode={tt['quantity'].mode().to_list()}, "
              f"avg_price={tt['price'].mean():.1f}")
else:
    print('No buyer/seller info in this round')

## Step 7: 订单间隔 / 成交间隔分布

In [70]:
order_intervals = order_interval_distribution(
    prices_long,
    product=PRODUCT,
    day=DAY,
    min_order_volume=MIN_ORDER_VOLUME if FILTER_BY_VOLUME else None,
)
trade_intervals = trade_interval_distribution(
    trades,
    product=PRODUCT,
    day=DAY,
    min_trade_quantity=MIN_TRADE_QUANTITY if FILTER_BY_VOLUME else None,
)

print(f"Order intervals count: {order_intervals.height}, mean={order_intervals['interval_ms'].mean() if order_intervals.height else 'NA'}")
print(f"Trade intervals count: {trade_intervals.height}, mean={trade_intervals['interval_ms'].mean() if trade_intervals.height else 'NA'}")

wm_overlay = wall_mid_df.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wm_overlay.columns:
    wm_overlay = wm_overlay.filter(pl.col('day') == DAY)

fig_o = plot_interval_distribution(
    order_intervals,
    title=f"Order Intervals {PRODUCT} Day {DAY}",
    wall_mid_df=wm_overlay.select(['timestamp', 'wall_mid']),
)
fig_o.show()

fig_t = plot_interval_distribution(
    trade_intervals,
    title=f"Trade Intervals {PRODUCT} Day {DAY}",
    wall_mid_df=wm_overlay.select(['timestamp', 'wall_mid']),
)
fig_t.show()

Order intervals count: 9982, mean=100.17030655179323
Trade intervals count: 334, mean=2979.940119760479


## Step 8: Wall Mid 奇偶统计

In [71]:
wm_stats = wall_mid_df.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wm_stats.columns:
    wm_stats = wm_stats.filter(pl.col('day') == DAY)
wm_stats = wm_stats.filter(pl.col('wall_mid').is_not_null()).with_columns(
    (2*pl.col('wall_mid') % 2 == 0).alias('is_even')
)

ratio_tbl = wm_stats.group_by('is_even').agg(pl.len().alias('n')).with_columns(
    (pl.col('n') / pl.col('n').sum()).alias('ratio')
).sort('is_even')

print('2 * Wall mid 奇偶占比:')
print(ratio_tbl)

trades_with_wm = (
    trades.filter(pl.col('product') == PRODUCT)
    .join(wm_stats.select(['timestamp', 'product', 'wall_mid', 'is_even']), on=['timestamp', 'product'], how='left')
    .filter(pl.col('is_even').is_not_null())
)

qty_col = 'quantity' if 'quantity' in trades_with_wm.columns else ('volume' if 'volume' in trades_with_wm.columns else None)
if qty_col is None:
    print('Trades 中没有 quantity/volume 列，无法统计成交量。')
else:
    trade_qty_tbl = (
        trades_with_wm.group_by('is_even')
        .agg([
            pl.col(qty_col).abs().sum().alias('total_trade_qty'),
            pl.len().alias('trade_count'),
        ])
        .sort('is_even')
    )
    print('\nWall mid 奇偶时的总成交量:')
    print(trade_qty_tbl)

even_ratio = ratio_tbl.filter(pl.col('is_even') == True)['ratio'].item() if ratio_tbl.filter(pl.col('is_even') == True).height else 0.0
odd_ratio = ratio_tbl.filter(pl.col('is_even') == False)['ratio'].item() if ratio_tbl.filter(pl.col('is_even') == False).height else 0.0
print(f"\nEven ratio={even_ratio:.4f}, Odd ratio={odd_ratio:.4f}, Δfrom 50%={abs(even_ratio-0.5):.4f}")

2 * Wall mid 奇偶占比:
shape: (2, 3)
┌─────────┬──────┬──────────┐
│ is_even ┆ n    ┆ ratio    │
│ ---     ┆ ---  ┆ ---      │
│ bool    ┆ u32  ┆ f64      │
╞═════════╪══════╪══════════╡
│ false   ┆ 4261 ┆ 0.462198 │
│ true    ┆ 4958 ┆ 0.537802 │
└─────────┴──────┴──────────┘

Wall mid 奇偶时的总成交量:
shape: (2, 3)
┌─────────┬─────────────────┬─────────────┐
│ is_even ┆ total_trade_qty ┆ trade_count │
│ ---     ┆ ---             ┆ ---         │
│ bool    ┆ i64             ┆ u32         │
╞═════════╪═════════════════╪═════════════╡
│ false   ┆ 715             ┆ 142         │
│ true    ┆ 991             ┆ 182         │
└─────────┴─────────────────┴─────────────┘

Even ratio=0.5378, Odd ratio=0.4622, Δfrom 50%=0.0378


## Step 9: 标准化档位成交统计

In [72]:
level_stats = normalized_level_trade_stats(
    trades=trades,
    wall_mid_df=wall_mid_df,
    product=PRODUCT,
    day=DAY,
    round_to=None,
)

print('标准化档位成交统计（norm_level = trade_price - wall_mid）:')
print(level_stats)

if level_stats.height:
    print('\n按成交次数排序 Top10:')
    print(level_stats.sort('trade_count', descending=True).head(10))

# 验证：norm_level 是否符合 0.5 网格，以及 .0/.5 占比
norm = level_stats.select(['norm_level', 'trade_count']).with_columns([
    (pl.col('norm_level') * 2).round(8).alias('norm_x2'),
]).with_columns([
    ((pl.col('norm_x2') - pl.col('norm_x2').round(0)).abs() < 1e-8).alias('is_half_grid'),
    (pl.col('norm_x2').round(0) % 2 == 0).alias('is_integer_level'),
])

total_trades = norm['trade_count'].sum() if norm.height else 0
on_grid_trades = norm.filter(pl.col('is_half_grid'))['trade_count'].sum() if norm.height else 0
int_trades = norm.filter(pl.col('is_half_grid') & pl.col('is_integer_level'))['trade_count'].sum() if norm.height else 0
half_trades = norm.filter(pl.col('is_half_grid') & (~pl.col('is_integer_level')))['trade_count'].sum() if norm.height else 0

print(f"\nHalf-grid integrity: {on_grid_trades}/{total_trades} = {on_grid_trades/total_trades if total_trades else 0:.4f}")
print(f"Integer-level trades (.0): {int_trades} ({int_trades/total_trades if total_trades else 0:.4f})")
print(f"Half-level trades (.5): {half_trades} ({half_trades/total_trades if total_trades else 0:.4f})")

标准化档位成交统计（norm_level = trade_price - wall_mid）:
shape: (33, 4)
┌────────────┬─────────────┬────────────────┬──────────────┐
│ norm_level ┆ trade_count ┆ total_quantity ┆ avg_quantity │
│ ---        ┆ ---         ┆ ---            ┆ ---          │
│ f64        ┆ u32         ┆ i64            ┆ f64          │
╞════════════╪═════════════╪════════════════╪══════════════╡
│ -9.5       ┆ 4           ┆ 21             ┆ 5.25         │
│ -9.0       ┆ 9           ┆ 44             ┆ 4.888889     │
│ -8.5       ┆ 1           ┆ 7              ┆ 7.0          │
│ -8.0       ┆ 7           ┆ 39             ┆ 5.571429     │
│ -7.5       ┆ 13          ┆ 69             ┆ 5.307692     │
│ …          ┆ …           ┆ …              ┆ …            │
│ 7.5        ┆ 11          ┆ 67             ┆ 6.090909     │
│ 8.0        ┆ 4           ┆ 23             ┆ 5.75         │
│ 8.5        ┆ 1           ┆ 5              ┆ 5.0          │
│ 9.0        ┆ 8           ┆ 39             ┆ 4.875        │
│ 9.5        ┆ 4      

## Step 10: Adverse Selection 检验

In [73]:
wm_series = wall_mid_df.filter(pl.col('product') == PRODUCT).select(['timestamp', 'wall_mid'])
as_result = adverse_selection(product_trades, wm_series)

print('Adverse Selection (taker PnL after trade):')
for h, v in as_result.items():
    label = 'SAFE (taker=noise)' if v <= 0 else 'DANGER (taker=informed)'
    print(f'  h={h:3d} steps: {v:+.4f}  {label}')

Adverse Selection (taker PnL after trade):
  h=  1 steps: +nan  DANGER (taker=informed)
  h=  5 steps: +nan  DANGER (taker=informed)
  h= 10 steps: +nan  DANGER (taker=informed)
  h= 50 steps: +nan  DANGER (taker=informed)


## 结论

在这里写你的观察...